In [1]:
import pandas as pd
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt

# Data Loading and First Look

In [2]:
# load the offenses file
df_offenses = pd.read_csv('../data/raw/INMATE_ACTIVE_OFFENSES_CPS.txt', sep='\t', encoding='latin-1')
df_offenses.head()

/var/folders/bz/l_61zlvj34gcp1c8znmr81rr0000gn/T/ipykernel_10850/3647353820.py:2: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df_offenses = pd.read_csv('../data/raw/INMATE_ACTIVE_OFFENSES_CPS.txt', sep='\t', encoding='latin-1')


,DCNumber,Sequence,OffenseDate,DateAdjudicated,County_of_Conviction,CaseNumber,prisonterm,adjudicationcharge_descr
0,132,1,11/22/2014,09/25/2018,DUVAL,1605872.0,170000,CONSPIR.TO TRAFF.DRUGS
1,132,2,11/22/2014,09/25/2018,DUVAL,1605872.0,150000,"HEROIN-SALE,MANUF/DELIVER"
2,132,3,11/24/2014,09/25/2018,DUVAL,1605872.0,170000,TRAFF ILL DRUGS 4-U/14 GRAMS
3,132,4,12/22/2014,09/25/2018,DUVAL,1605872.0,170000,"TRAFF HER.,ETC.14-U/28 GR"
4,132,5,02/10/2015,09/25/2018,DUVAL,1605872.0,170000,TRAFF HER.ETC 28G-U/30 KG


In [3]:
df = pd.read_csv('../data/raw/INMATE_ACTIVE_ROOT.txt', sep='\t', encoding='latin-1')
df.head()

,DCNumber,LastName,FirstName,MiddleName,NameSuffix,Race,Sex,BirthDate,PrisonReleaseDate,ReceiptDate,releasedateflag_descr,race_descr,custody_description,FACILITY_description
0,000132,TELFAIR,MICHAEL,NaN,NaN,B,M,12/04/1967,01/27/2033,10/08/2018,valid release date,BLACK,MINIMUM,MADISON C.I.
1,000155,LOCKETT,JERRY,I,NaN,B,M,09/26/1985,05/14/2026,05/23/2024,valid release date,BLACK,MEDIUM,GULF C.I.
2,000175,ELMORE,CHAD,R,NaN,W,M,10/20/1982,07/20/2028,06/11/2025,valid release date,WHITE,CLOSE,JACKSON C.I.
3,000191,CARRILLO,ISAIAS,NaN,JR,W,M,02/21/1998,03/30/2028,09/24/2025,valid release date,WHITE,MEDIUM,APALACHEE EAST UNIT
4,000203,RHINESMITH,AUSTIN,A,NaN,W,M,09/12/2000,09/06/2027,07/24/2025,valid release date,WHITE,MEDIUM,CFRC-EAST


# Standardizing DC Number

In [4]:
# standardize DCNumber in both dataframes
df_offenses['DCNumber'] = df_offenses['DCNumber'].astype(str).str.zfill(6)

# verify the formats match
print("Inmate file DCNumber sample:", df['DCNumber'].head(3).tolist())
print("Offenses file DCNumber sample:", df_offenses['DCNumber'].head(3).tolist())

Inmate file DCNumber sample: ['000132', '000155', '000175']
Offenses file DCNumber sample: ['000132', '000132', '000132']


# Comparing Shapes

In [6]:
# compare shapes
print("Shape of INMATE_ACTIVE_ROOT df:")
print(df.shape)

print("Shape of INMATE_ACTIVE_OFFENSES_CPS df:")
print(df_offenses.shape)

Shape of INMATE_ACTIVE_ROOT df:
(90663, 14)
Shape of INMATE_ACTIVE_OFFENSES_CPS df:
(373175, 8)


In [17]:
# compare inmate counts between the two files
df_inmates = df['DCNumber'].nunique()
df_offenses_inmates = df_offenses['DCNumber'].nunique()

print(f"Unique inmates in INMATE_ACTIVE_ROOT: {df_inmates}")
print(f"Unique inmates in INMATE_ACTIVE_OFFENSES_CPS: {df_offenses_inmates}")
print(f"Difference: {df_inmates - df_offenses_inmates}")

# check if there are any inmates in df that are NOT in df_offenses
in_root_not_offenses = set(df['DCNumber']) - set(df_offenses['DCNumber'])
print(f"\nInmates in ROOT but not in OFFENSES: {len(in_root_not_offenses)}")

# check if there are any inmates in df_offenses that are NOT in df
in_offenses_not_root = set(df_offenses['DCNumber']) - set(df['DCNumber'])
print(f"Inmates in OFFENSES but not in ROOT: {len(in_offenses_not_root)}")

Unique inmates in INMATE_ACTIVE_ROOT: 90663
Unique inmates in INMATE_ACTIVE_OFFENSES_CPS: 90650
Difference: 13

Inmates in ROOT but not in OFFENSES: 13
Inmates in OFFENSES but not in ROOT: 0


In [18]:
# check if any of the 13 unmatched inmates are at Columbia Annex
unmatched = set(df['DCNumber']) - set(df_offenses['DCNumber'])
unmatched_columbia = columbia_annex[columbia_annex['DCNumber'].isin(unmatched)]
print(f"Unmatched inmates at Columbia Annex: {len(unmatched_columbia)}")

Unmatched inmates at Columbia Annex: 0


# Filter and Join Dataframes

In [7]:
# filter inmate df to Columbia Annex only
columbia_annex = df[df['FACILITY_description'] == 'COLUMBIA ANNEX']
print(f"Inmates at Columbia Annex: {len(columbia_annex)}")

Inmates at Columbia Annex: 1467


## Join Strategy: Left Join on DCNumber

To answer this question, we need to combine data from two files:
- `INMATE_ACTIVE_ROOT.txt` — contains inmate demographic and facility information
- `INMATE_ACTIVE_OFFENSES_CPS.txt` — contains offense records including county of conviction

The two files are linked by `DCNumber`, a unique alphanumeric identifier assigned to
each inmate.

We use a **left join**, which works as follows: for each row in the left table
(the Columbia Annex inmates), we look up the `DCNumber` in the right table
(the offenses file) and attach all matching offense records. If an inmate has
multiple offenses, they will appear as multiple rows in the joined table.
If an inmate has no matching offense record, they are retained in the result
with `NaN` for offense columns.

A left join is the correct choice here because we want to preserve all Columbia
Annex inmates regardless of whether they have a matching offense record. A inner
join would silently drop any inmates with no offense records, potentially
undercounting the population. In our case, we verified that all Columbia Annex
inmates have matching offense records, so the choice of join type does not affect
the result — but the left join is still the safer and more principled approach.

In [9]:
# join with offenses file on DCNumber
columbia_offenses = columbia_annex.merge(df_offenses, on='DCNumber', how='left')
columbia_offenses.head()

,DCNumber,LastName,FirstName,MiddleName,NameSuffix,Race,Sex,BirthDate,PrisonReleaseDate,ReceiptDate,...,race_descr,custody_description,FACILITY_description,Sequence,OffenseDate,DateAdjudicated,County_of_Conviction,CaseNumber,prisonterm,adjudicationcharge_descr
0,000308,BOLTER,JORDAN,L,NaN,B,M,10/15/1994,12/19/2028,05/29/2024,...,BLACK,CLOSE,COLUMBIA ANNEX,2,01/28/2024,05/07/2024,HILLSBOROUGH,2401469.0,50000,ROBB. GUN OR DEADLY WPN
1,000308,BOLTER,JORDAN,L,NaN,B,M,10/15/1994,12/19/2028,05/29/2024,...,BLACK,CLOSE,COLUMBIA ANNEX,3,01/28/2024,05/07/2024,HILLSBOROUGH,2401469.0,50000,AGG BATTERY INTENDED HARM
2,000308,BOLTER,JORDAN,L,NaN,B,M,10/15/1994,12/19/2028,05/29/2024,...,BLACK,CLOSE,COLUMBIA ANNEX,4,01/28/2024,05/07/2024,HILLSBOROUGH,2401469.0,50000,AGG BATTERY/W/DEADLY WEAPON
3,011344,JONES,IVAN,D,NaN,B,M,07/08/2003,12/06/2029,12/02/2021,...,BLACK,CLOSE,COLUMBIA ANNEX,1,07/22/2020,09/17/2021,HILLSBOROUGH,2010303.0,100000,ROBB. GUN OR DEADLY WPN
4,011344,JONES,IVAN,D,NaN,B,M,07/08/2003,12/06/2029,12/02/2021,...,BLACK,CLOSE,COLUMBIA ANNEX,2,07/22/2020,09/17/2021,HILLSBOROUGH,2010303.0,100000,BURGLARY ASSAULT ANY PERSON


In [10]:
columbia_offenses.shape

(6087, 21)

In [14]:
# count by county of conviction
county_counts = columbia_offenses.groupby('County_of_Conviction')['DCNumber'].nunique().reset_index()
county_counts.columns = ['County_of_Conviction', 'Inmate_Count']
county_counts = county_counts.sort_values('Inmate_Count', ascending=False)

print(f"\nTotal counties: {len(county_counts)}")
print(county_counts.to_string())


Total counties: 66
   County_of_Conviction  Inmate_Count
14                DUVAL           167
27         HILLSBOROUGH           128
47               ORANGE            99
42           MIAMI-DADE            87
5               BROWARD            84
51             PINELLAS            81
63              VOLUSIA            80
40               MARION            61
52                 POLK            61
49           PALM BEACH            52
35                  LEE            42
15             ESCAMBIA            38
39              MANATEE            38
50                PASCO            36
0               ALACHUA            33
56             SEMINOLE            33
55             SARASOTA            31
36                 LEON            29
58            ST. LUCIE            29
8                CITRUS            25
53               PUTNAM            25
4               BREVARD            25
34                 LAKE            21
2                   BAY            21
41               MARTIN       

In [16]:
# sanity check 1: total unique inmates at Columbia Annex
print(f"Unique inmates at Columbia Annex (from df): {len(columbia_annex)}")
print(f"Sum of county counts: {county_counts['Inmate_Count'].sum()}")

# sanity check 2: any inmates with no county match (nulls)?
null_county = columbia_offenses[columbia_offenses['County_of_Conviction'].isnull()]['DCNumber'].nunique()
print(f"Inmates with no county match: {null_county}")

# sanity check 3: any inmates counted in multiple counties?
inmates_per_county = columbia_offenses.groupby('DCNumber')['County_of_Conviction'].nunique()
multi_county = (inmates_per_county > 1).sum()
print(f"Inmates with offenses in multiple counties: {multi_county}")

Unique inmates at Columbia Annex (from df): 1467
Sum of county counts: 1586
Inmates with no county match: 0
Inmates with offenses in multiple counties: 109
